In [6]:
!pip install --upgrade openpyxl


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd

In [9]:
import sys
print(sys.executable)

c:\Users\Acer\anaconda3\python.exe


In [10]:
import sys
!{sys.executable} -m pip install --upgrade openpyxl


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
df_2009 = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df_2009, df_2010], ignore_index=True)
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [12]:
df.info()
print("\nNull counts:")
print(df.isnull().sum())
print("\nCancelled invoices (start with 'C'):")
print(df['Invoice'].astype(str).str.startswith('C').sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB

Null counts:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Cancelled invoices (start with 'C'):
19494


In [17]:
# --- Cleaning Step 1: Separate cancellations/returns ---
df['Invoice'] = df['Invoice'].astype(str)
df_returns = df[df['Invoice'].str.startswith('C')].copy()
df_sales = df[~df['Invoice'].str.startswith('C')].copy()
df_sales[df_sales['Quantity'] <= 0].head(10)

print("Sales rows:", df_sales.shape[0])
print("Returns rows:", df_returns.shape[0])

Sales rows: 1047877
Returns rows: 19494


In [27]:
# --- Cleaning Step 2: Drop rows with missing Customer ID or Description ---
before = df_sales.shape[0]

df_sales = df_sales.dropna(subset=['Customer ID', 'Description'])
df_sales[df_sales['Price'] <= 0].head(10)

after = df_sales.shape[0]
print(f"Dropped {before - after} rows ({(before - after)/before:.1%})")
print("Remaining rows:", after)

Dropped 0 rows (0.0%)
Remaining rows: 805620


In [28]:
# --- Cleaning Step 3: Check for negative/zero quantity or price in sales data ---
df_sales[df_sales.duplicated(keep=False)].sort_values('Invoice').head(10)
print("Negative quantity in sales:", (df_sales['Quantity'] <= 0).sum())
print("Zero or negative price in sales:", (df_sales['Price'] <= 0).sum())
print("Duplicate rows:", df_sales.duplicated().sum())
df_sales[df_sales.duplicated(keep=False)].sort_values('Invoice').head(10)

Negative quantity in sales: 0
Zero or negative price in sales: 71
Duplicate rows: 26125


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
390,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329.0,United Kingdom
388,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom


In [29]:
# --- Cleaning Step 4: Drop zero/negative price rows and true duplicates ---
before = df_sales.shape[0]

df_sales = df_sales[df_sales['Price'] > 0]
df_sales = df_sales.drop_duplicates()

after = df_sales.shape[0]
print(f"Dropped {before - after} rows ({(before - after)/before:.2%})")
print("Final clean row count:", after)

Dropped 26195 rows (3.25%)
Final clean row count: 779425


In [30]:
# --- Cleaning Step 5: Clean column names ---
df_sales = df_sales.rename(columns={'Customer ID': 'CustomerID'})
df_sales.columns = df_sales.columns.str.strip()
print(df_sales.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']


In [31]:
# --- Save cleaned data ---
df_sales.to_csv('../data/cleaned_sales.csv', index=False)
df_returns.to_csv('../data/returns.csv', index=False)
print("Saved cleaned_sales.csv and returns.csv to /data")

Saved cleaned_sales.csv and returns.csv to /data
